# Baseline Forecasting Notebook

This notebook prepares the electricity load data and builds several baseline forecasting models for comparison. It includes naive persistence methods, seasonal baselines, LSTM point forecasts, a probabilistic LSTM, and a Holt-Winters statistical baseline. The resulting predictions are saved for downstream evaluation against Transformer-based models.


## Environment and Library Setup

Imports the core Python, data processing, visualization, and PyTorch libraries required for the baseline forecasting workflow.


In [4]:
from __future__ import annotations
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Tuple, Optional
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader,TensorDataset
import zipfile
import requests
import io
print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

PyTorch version: 2.10.0+cu128
Using device: cuda


## Dataset Loading

Defines the data loading routine and standardizes the timestamp column so the electricity load data can be indexed consistently.


In [5]:
def load_electricity_data():
  # Step 1: Download zip
  url = "https://archive.ics.uci.edu/static/public/321/electricityloaddiagrams20112014.zip"
  response = requests.get(url)

  # Step 2: Extract zip in memory
  z = zipfile.ZipFile(io.BytesIO(response.content))

  # Step 3: Read the txt file inside zip
  file_name = z.namelist()[0]   # should be LD2011_2014.txt

  df = pd.read_csv(
      z.open(file_name),
      sep=";",
      decimal=",",
      parse_dates=[0],
      quotechar='"'
  )

  # Step 4: Fix datetime column
  df.rename(columns={df.columns[0]: "datetime"}, inplace=True)
  df.set_index("datetime", inplace=True)
  return df

## Dataset Import

Loads the raw electricity dataset into memory for preprocessing and feature engineering.


In [6]:
df = load_electricity_data()

## Hourly Aggregation

Aggregates the raw observations to an hourly frequency, creating the regular time series used by all forecasting models.


In [7]:
def aggregate_data(df):
  df["total_load"] = df.sum(axis=1)
  df_hourly = df["total_load"].resample("H").sum().to_frame()
  df_hourly.head()
  return df_hourly
df_hourly = aggregate_data(df)

/tmp/ipykernel_1353/3915923716.py:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df["total_load"].resample("H").sum().to_frame()


## Calendar Feature Engineering

Creates cyclical calendar features for hour, day of week, and month so the models can learn recurring temporal patterns.


In [8]:
def create_features(df_hourly):
  # If df_hourly is a Series, convert it to a DataFrame first
  if isinstance(df_hourly, pd.Series):
      df_hourly = df_hourly.to_frame(name="total_load")

  # Make sure index is datetime
  df_hourly.index = pd.to_datetime(df_hourly.index)

  # Create a copy so original stays untouched
  df_features = df_hourly.copy()

  # Basic calendar/time features
  df_features["hour_of_day"] = df_features.index.hour
  df_features["day_of_week"] = df_features.index.dayofweek   # Monday=0, Sunday=6
  df_features["is_weekend"] = (df_features["day_of_week"] >= 5).astype(int)
  df_features["month"] = df_features.index.month

  # Optional cyclical encoding
  df_features["hour_sin"] = np.sin(2 * np.pi * df_features["hour_of_day"] / 24)
  df_features["hour_cos"] = np.cos(2 * np.pi * df_features["hour_of_day"] / 24)

  df_features["dayofweek_sin"] = np.sin(2 * np.pi * df_features["day_of_week"] / 7)
  df_features["dayofweek_cos"] = np.cos(2 * np.pi * df_features["day_of_week"] / 7)

  df_features["month_sin"] = np.sin(2 * np.pi * (df_features["month"] - 1) / 12)
  df_features["month_cos"] = np.cos(2 * np.pi * (df_features["month"] - 1) / 12)

  return df_features

## Feature Construction

Applies the feature engineering routine and prepares the enriched dataset for validation and modeling.


In [9]:
df_features = create_features(df_hourly)

## Outlier Diagnostics

Uses the interquartile range method to summarize potential load outliers without removing observations from the dataset.


In [10]:
def check_outliers_iqr(df, target_col="total_load"):
    """
    Checks outliers in a numeric column using IQR rule.
    Returns summary dictionary and dataframe of outlier rows.
    """
    series = df[target_col].dropna()

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (series < lower_bound) | (series > upper_bound)
    outliers = df.loc[outlier_mask]

    summary = {
        "target_col": target_col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "num_outliers": int(outlier_mask.sum()),
        "pct_outliers": round(100 * outlier_mask.mean(), 2)
    }

    return summary, outliers
summary, outliers = check_outliers_iqr(df_hourly, target_col="total_load")

## Chronological Data Split

Defines a time-aware train, validation, and test split that preserves chronological order and prevents future leakage.


In [11]:
def split_time_series(df, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
    """
    Splits dataframe chronologically into train, val, test.
    Ratios must sum to 1.
    """
    if not np.isclose(train_ratio + val_ratio + test_ratio, 1.0):
        raise ValueError("train_ratio + val_ratio + test_ratio must equal 1.0")

    n = len(df)
    train_end = int(n * train_ratio)
    val_end = train_end + int(n * val_ratio)

    train_df = df.iloc[:train_end].copy()
    val_df = df.iloc[train_end:val_end].copy()
    test_df = df.iloc[val_end:].copy()

    return train_df, val_df, test_df

## Dataset Partitioning

Applies the chronological split to create the train, validation, and test datasets used throughout the experiments.


In [12]:
train_df, val_df, test_df = split_time_series(df_features)

## Normalization Utility

Defines the scaling routine that fits normalization parameters on the training set and applies them consistently to all splits.


In [13]:
def normalize_splits(train_df, val_df, test_df, feature_cols):
    """
    Fits StandardScaler on train_df[feature_cols] only,
    then transforms train/val/test using the same scaler.

    Returns scaled dataframes and fitted scaler.
    """
    scaler = StandardScaler()

    train_scaled = train_df.copy()
    val_scaled = val_df.copy()
    test_scaled = test_df.copy()

    train_scaled[feature_cols] = scaler.fit_transform(train_df[feature_cols])
    val_scaled[feature_cols] = scaler.transform(val_df[feature_cols])
    test_scaled[feature_cols] = scaler.transform(test_df[feature_cols])

    return train_scaled, val_scaled, test_scaled, scaler

## Feature Scaling

Selects the model input features, scales the dataset partitions, and prepares normalized inputs for sequence generation.


In [14]:
feature_cols_to_scale = [
    "total_load"
]
train_scaled, val_scaled, test_scaled, scaler = normalize_splits(
    train_df, val_df, test_df, feature_cols_to_scale
)

## Forecasting Configuration and Sequence Utilities

Defines the input window, forecast horizon, model feature set, sequence builder, and random seed helper used by the baseline models.


In [15]:

INPUT_WINDOW = 168
FORECAST_HORIZON = 24
BATCH_SIZE = 64

FEATURE_COLS = [
    'total_load',
    'hour_sin', 'hour_cos',
    'dayofweek_sin', 'dayofweek_cos',
    'month_sin', 'month_cos'
]


def create_sequences(data, input_window=INPUT_WINDOW, forecast_horizon=FORECAST_HORIZON):
    features = data[FEATURE_COLS].values
    targets = data['total_load'].values

    X, y = [], []
    for i in range(len(features) - input_window - forecast_horizon + 1):
        X.append(features[i:i + input_window])
        y.append(targets[i + input_window:i + input_window + forecast_horizon])

    return (
        torch.tensor(np.array(X), dtype=torch.float32),
        torch.tensor(np.array(y), dtype=torch.float32)
    )


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)



## Train, Validation, and Test Sequence Creation

Transforms each normalized split into supervised learning tensors and stores the scaled test targets for evaluation.


In [16]:
X_train, y_train = create_sequences(train_scaled)
X_val, y_val = create_sequences(val_scaled)
X_test, y_test = create_sequences(test_scaled)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape},   y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")

# Store actual test targets for future use
y_test_scaled = y_test.numpy()


X_train: torch.Size([24354, 168, 7]), y_train: torch.Size([24354, 24])
X_val:   torch.Size([5068, 168, 7]),   y_val:   torch.Size([5068, 24])
X_test:  torch.Size([5070, 168, 7]),  y_test:  torch.Size([5070, 24])


## Naive and Seasonal Forecast Generation

Builds persistence-based forecasting baselines and generates daily and weekly seasonal predictions for the test set.


In [17]:
def generate_persistence_forecast(X_test, forecast_horizon=FORECAST_HORIZON):
    """Naive persistence: next 24h = last 24h of input window."""
    naive_persistence_preds_scaled = X_test[:, -forecast_horizon:, 0].numpy()
    return naive_persistence_preds_scaled


def seasonal_naive_forecast(X_context_scaled, forecast_horizon, seasonal_lag):
    """
    Seasonal naive forecast.
    For forecast step h, copy value from (-seasonal_lag + h) of input window.
    """
    load_col = X_context_scaled[:, :, 0].numpy()
    preds = np.zeros((load_col.shape[0], forecast_horizon))

    for h in range(forecast_horizon):
        idx = -seasonal_lag + h
        preds[:, h] = load_col[:, idx]

    return preds


# Naive persistence forecast on test set
naive_persistence_preds_scaled = generate_persistence_forecast(
    X_test, forecast_horizon=FORECAST_HORIZON
)

# Seasonal naive day forecast on test set
seasonal_naive_day_preds_scaled = seasonal_naive_forecast(
    X_test, FORECAST_HORIZON, seasonal_lag=24
)

# Seasonal naive week forecast on test set
seasonal_naive_week_preds_scaled = seasonal_naive_forecast(
    X_test, FORECAST_HORIZON, seasonal_lag=168
)


## LSTM Baseline Architectures

Defines the univariate and multivariate LSTM architectures used as neural point-forecasting baselines.


In [18]:
class LSTMForecaster(nn.Module):
    """Univariate LSTM."""
    def __init__(self, input_size=1, hidden_size=64, num_layers=2,
                 dropout=0.1, forecast_horizon=24):
        super().__init__()
        lstm_dropout = dropout if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=lstm_dropout,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, forecast_horizon)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]
        last_hidden = self.dropout(last_hidden)
        return self.fc(last_hidden)


class MultiFeatureLSTMForecaster(nn.Module):
    """Multivariate LSTM."""
    def __init__(self, input_size=7, hidden_size=64, num_layers=2,
                 dropout=0.1, forecast_horizon=24):
        super().__init__()
        lstm_dropout = dropout if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=lstm_dropout,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, forecast_horizon)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]
        last_hidden = self.dropout(last_hidden)
        return self.fc(last_hidden)


## Probabilistic LSTM Architecture

Defines the multivariate LSTM variant that outputs q10, q50, and q90 forecasts for each prediction horizon.


In [21]:
class ProbabilisticLSTM(nn.Module):
    """
    Multivariate LSTM that outputs quantile predictions.
    Architecture mirrors the existing MultiFeatureLSTMForecaster
    but with an output head producing [batch, 24, 3] instead of [batch, 24].
    """
    def __init__(self, input_size=7, hidden_size=64, num_layers=2,
                 dropout=0.1, forecast_horizon=24, num_quantiles=3):
        super().__init__()
        self.forecast_horizon = forecast_horizon
        self.num_quantiles = num_quantiles
        lstm_dropout = dropout if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=lstm_dropout,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, forecast_horizon * num_quantiles)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_hidden = self.dropout(out[:, -1, :])
        output = self.fc(last_hidden)
        return output.reshape(-1, self.forecast_horizon, self.num_quantiles)



# Skip these cells if you are using the pt files to generate the predicitons

## LSTM Training and Validation Steps

Provides shared routines for one training epoch and validation loss evaluation during LSTM fitting.


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, dev):
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(dev), y_batch.to(dev)

        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate_loss(model, loader, criterion, dev):
    model.eval()
    total_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(dev), y_batch.to(dev)
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(loader.dataset)


## LSTM Prediction Helper

Collects model predictions and corresponding target arrays from a data loader after training is complete.


In [ ]:
@torch.no_grad()
def predict_model(model, loader, dev):
    """Return predictions and targets from a dataloader."""
    model.eval()
    preds_all, y_all = [], []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(dev)
        preds = model(X_batch).cpu().numpy()
        preds_all.append(preds)
        y_all.append(y_batch.numpy())

    return np.vstack(preds_all), np.vstack(y_all)


## LSTM Training Loop

Trains an LSTM model with validation monitoring, early stopping, and restoration of the best observed weights.


In [ ]:
def train_lstm(model, train_loader, val_loader, num_epochs=20, lr=1e-3,
               patience=5, dev='cpu', label="LSTM"):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_state = None
    wait = 0

    print(f"\n===== Training {label} =====")
    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, dev)
        val_loss = evaluate_loss(model, val_loader, criterion, dev)

        print(f"{label} | Epoch {epoch:02d}/{num_epochs} | "
              f"Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            print(f"{label} | Early stopping at epoch {epoch}.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"{label} | Best Val Loss: {best_val_loss:.6f}")
    return model


## Univariate LSTM Forecasting

Trains the load-only LSTM baseline and generates its scaled predictions on the test set.


In [ ]:
set_seed(42)

X_train_uni = X_train[:, :, 0:1]
X_val_uni = X_val[:, :, 0:1]
X_test_uni = X_test[:, :, 0:1]

train_loader_uni = DataLoader(TensorDataset(X_train_uni, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader_uni = DataLoader(TensorDataset(X_val_uni, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader_uni = DataLoader(TensorDataset(X_test_uni, y_test), batch_size=BATCH_SIZE, shuffle=False)

model_uni = LSTMForecaster(
    input_size=1,
    hidden_size=64,
    num_layers=2,
    dropout=0.1,
    forecast_horizon=FORECAST_HORIZON
).to(device)

model_uni = train_lstm(
    model_uni,
    train_loader_uni,
    val_loader_uni,
    num_epochs=20,
    lr=1e-3,
    patience=5,
    dev=device,
    label="Univariate LSTM"
)

univariate_lstm_preds_scaled, univariate_lstm_targets_scaled = predict_model(
    model_uni, test_loader_uni, device
)



===== Training Univariate LSTM =====
Univariate LSTM | Epoch 01/20 | Train Loss: 0.183404 | Val Loss: 0.033081
Univariate LSTM | Epoch 02/20 | Train Loss: 0.037266 | Val Loss: 0.026921
Univariate LSTM | Epoch 03/20 | Train Loss: 0.032014 | Val Loss: 0.025289
Univariate LSTM | Epoch 04/20 | Train Loss: 0.029613 | Val Loss: 0.024240
Univariate LSTM | Epoch 05/20 | Train Loss: 0.028333 | Val Loss: 0.024079
Univariate LSTM | Epoch 06/20 | Train Loss: 0.027604 | Val Loss: 0.026595
Univariate LSTM | Epoch 07/20 | Train Loss: 0.026728 | Val Loss: 0.022056
Univariate LSTM | Epoch 08/20 | Train Loss: 0.026117 | Val Loss: 0.021634
Univariate LSTM | Epoch 09/20 | Train Loss: 0.025976 | Val Loss: 0.021590
Univariate LSTM | Epoch 10/20 | Train Loss: 0.025367 | Val Loss: 0.021193
Univariate LSTM | Epoch 11/20 | Train Loss: 0.025214 | Val Loss: 0.020422
Univariate LSTM | Epoch 12/20 | Train Loss: 0.024933 | Val Loss: 0.019787
Univariate LSTM | Epoch 13/20 | Train Loss: 0.024638 | Val Loss: 0.019693


## Multivariate LSTM Forecasting

Trains the feature-rich LSTM baseline and generates its scaled predictions on the test set.


In [ ]:
set_seed(42)

train_loader_multi = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader_multi = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader_multi = DataLoader(TensorDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

model_multi = MultiFeatureLSTMForecaster(
    input_size=len(FEATURE_COLS),
    hidden_size=64,
    num_layers=2,
    dropout=0.1,
    forecast_horizon=FORECAST_HORIZON
).to(device)

model_multi = train_lstm(
    model_multi,
    train_loader_multi,
    val_loader_multi,
    num_epochs=20,
    lr=1e-3,
    patience=5,
    dev=device,
    label="Multivariate LSTM"
)

multivariate_lstm_preds_scaled, multivariate_lstm_targets_scaled = predict_model(
    model_multi, test_loader_multi, device
)



===== Training Multivariate LSTM =====
Multivariate LSTM | Epoch 01/20 | Train Loss: 0.136334 | Val Loss: 0.030536
Multivariate LSTM | Epoch 02/20 | Train Loss: 0.033112 | Val Loss: 0.025903
Multivariate LSTM | Epoch 03/20 | Train Loss: 0.029529 | Val Loss: 0.025520
Multivariate LSTM | Epoch 04/20 | Train Loss: 0.027519 | Val Loss: 0.023623
Multivariate LSTM | Epoch 05/20 | Train Loss: 0.026253 | Val Loss: 0.022643
Multivariate LSTM | Epoch 06/20 | Train Loss: 0.025441 | Val Loss: 0.020666
Multivariate LSTM | Epoch 07/20 | Train Loss: 0.024756 | Val Loss: 0.019089
Multivariate LSTM | Epoch 08/20 | Train Loss: 0.023989 | Val Loss: 0.021864
Multivariate LSTM | Epoch 09/20 | Train Loss: 0.023182 | Val Loss: 0.018786
Multivariate LSTM | Epoch 10/20 | Train Loss: 0.023714 | Val Loss: 0.028267
Multivariate LSTM | Epoch 11/20 | Train Loss: 0.023741 | Val Loss: 0.022339
Multivariate LSTM | Epoch 12/20 | Train Loss: 0.022752 | Val Loss: 0.019902
Multivariate LSTM | Epoch 13/20 | Train Loss: 0.

## Stored Baseline Output Summary

Prints the key arrays produced by the baseline workflow so they can be referenced in later evaluation steps.


In [ ]:
print("\nStored prediction variables ready for future use:")

print("Actual test targets:")
print(" - y_test_scaled")

print("\nNaive / Seasonal predictions:")
print(" - naive_persistence_preds_scaled")
print(" - seasonal_naive_day_preds_scaled")
print(" - seasonal_naive_week_preds_scaled")

print("\nLSTM predictions:")
print(" - univariate_lstm_preds_scaled")
print(" - multivariate_lstm_preds_scaled")

print("\nOptional matching target arrays returned alongside LSTM predictions:")
print(" - univariate_lstm_targets_scaled")
print(" - multivariate_lstm_targets_scaled")


Stored prediction variables ready for future use:
Actual test targets:
 - y_test_scaled

Naive / Seasonal predictions:
 - naive_persistence_preds_scaled
 - seasonal_naive_day_preds_scaled
 - seasonal_naive_week_preds_scaled

LSTM predictions:
 - univariate_lstm_preds_scaled
 - multivariate_lstm_preds_scaled

Optional matching target arrays returned alongside LSTM predictions:
 - univariate_lstm_targets_scaled
 - multivariate_lstm_targets_scaled


## Quantile Loss Definition

Defines the pinball loss function used to optimize probabilistic forecasts across the selected quantiles.


In [ ]:

class QuantileLossLSTM(nn.Module):
    """Pinball loss for probabilistic LSTM — identical to Transformer's loss."""
    def __init__(self, quantiles=[0.1, 0.5, 0.9]):
        super().__init__()
        self.quantiles = quantiles

    def forward(self, predictions, targets):
        # predictions: [batch, 24, 3]
        # targets: [batch, 24]
        targets = targets.unsqueeze(-1)
        losses = []
        for i, q in enumerate(self.quantiles):
            errors = targets - predictions[:, :, i].unsqueeze(-1)
            loss = torch.max(q * errors, (q - 1) * errors)
            losses.append(loss.mean())
        return sum(losses) / len(losses)


## Probabilistic LSTM Training Loop

Trains the probabilistic LSTM with quantile loss, validation monitoring, gradient clipping, and early stopping.


In [ ]:
def train_probabilistic_lstm(model, train_loader, val_loader, num_epochs=20,
                              lr=1e-3, patience=5, dev='cpu', label="Prob. LSTM"):
    """
    Training loop for the Probabilistic LSTM.
    Uses quantile loss instead of MSE, with early stopping
    and gradient clipping for stability.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = QuantileLossLSTM(quantiles=[0.1, 0.5, 0.9])

    best_val_loss = float('inf')
    best_state = None
    wait = 0

    print(f"\n===== Training {label} =====")
    for epoch in range(1, num_epochs + 1):
        # --- Training ---
        model.train()
        total_train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(dev), y_batch.to(dev)
            optimizer.zero_grad()
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_train_loss += loss.item() * X_batch.size(0)

        avg_train_loss = total_train_loss / len(train_loader.dataset)

        # --- Validation ---
        model.eval()
        total_val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(dev), y_batch.to(dev)
                preds = model(X_batch)
                loss = criterion(preds, y_batch)
                total_val_loss += loss.item() * X_batch.size(0)

        avg_val_loss = total_val_loss / len(val_loader.dataset)

        print(f"{label} | Epoch {epoch:02d}/{num_epochs} | "
              f"Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")

        # --- Early stopping ---
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            print(f"{label} | Early stopping at epoch {epoch}.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"{label} | Best Val Loss: {best_val_loss:.6f}")
    return model


## Probabilistic LSTM Prediction Helper

Collects probabilistic LSTM quantile predictions and matching target values from the test data loader.


In [ ]:
@torch.no_grad()
def predict_probabilistic_model(model, loader, dev):
    """
    Returns quantile predictions [N, 24, 3] and targets [N, 24]
    from a dataloader for any model outputting quantiles.
    """
    model.eval()
    preds_all, y_all = [], []
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(dev)
        preds = model(X_batch).cpu().numpy()
        preds_all.append(preds)
        y_all.append(y_batch.numpy())
    return np.vstack(preds_all), np.vstack(y_all)


## Probabilistic LSTM Forecasting

Initializes, trains, and validates the probabilistic LSTM using the multivariate sequence data.


In [ ]:
set_seed(42)

model_prob_lstm = ProbabilisticLSTM(
    input_size=len(FEATURE_COLS),
    hidden_size=64,
    num_layers=2,
    dropout=0.1,
    forecast_horizon=FORECAST_HORIZON,
    num_quantiles=3
).to(device)

model_prob_lstm = train_probabilistic_lstm(
    model_prob_lstm,
    train_loader_multi,
    val_loader_multi,
    num_epochs=20,
    lr=1e-3,
    patience=5,
    dev=device,
    label="Probabilistic LSTM"
)



===== Training Probabilistic LSTM =====
Probabilistic LSTM | Epoch 01/20 | Train Loss: 0.104949 | Val Loss: 0.037268
Probabilistic LSTM | Epoch 02/20 | Train Loss: 0.043627 | Val Loss: 0.033424
Probabilistic LSTM | Epoch 03/20 | Train Loss: 0.039909 | Val Loss: 0.033107
Probabilistic LSTM | Epoch 04/20 | Train Loss: 0.037995 | Val Loss: 0.031098
Probabilistic LSTM | Epoch 05/20 | Train Loss: 0.036762 | Val Loss: 0.032208
Probabilistic LSTM | Epoch 06/20 | Train Loss: 0.036026 | Val Loss: 0.031871
Probabilistic LSTM | Epoch 07/20 | Train Loss: 0.035100 | Val Loss: 0.031471
Probabilistic LSTM | Epoch 08/20 | Train Loss: 0.034529 | Val Loss: 0.030436
Probabilistic LSTM | Epoch 09/20 | Train Loss: 0.034137 | Val Loss: 0.029307
Probabilistic LSTM | Epoch 10/20 | Train Loss: 0.033629 | Val Loss: 0.029734
Probabilistic LSTM | Epoch 11/20 | Train Loss: 0.033361 | Val Loss: 0.028183
Probabilistic LSTM | Epoch 12/20 | Train Loss: 0.032999 | Val Loss: 0.028425
Probabilistic LSTM | Epoch 13/20 | 

## Probabilistic LSTM Prediction Outputs

Generates the probabilistic LSTM test predictions and separates the quantile outputs for evaluation.


In [ ]:
prob_lstm_all_preds_scaled, prob_lstm_targets_scaled = predict_probabilistic_model(
    model_prob_lstm, test_loader_multi, device
)

# --- Extract individual quantiles ---
prob_lstm_q10_scaled = prob_lstm_all_preds_scaled[:, :, 0]
prob_lstm_q50_scaled = prob_lstm_all_preds_scaled[:, :, 1]
prob_lstm_q90_scaled = prob_lstm_all_preds_scaled[:, :, 2]

print(f"\nProbabilistic LSTM predictions shape: {prob_lstm_all_preds_scaled.shape}")
print("Stored variables:")
print(" - prob_lstm_all_preds_scaled  [N, 24, 3]")
print(" - prob_lstm_q10_scaled        [N, 24]")
print(" - prob_lstm_q50_scaled        [N, 24]")
print(" - prob_lstm_q90_scaled        [N, 24]")


Probabilistic LSTM predictions shape: (5070, 24, 3)
Stored variables:
 - prob_lstm_all_preds_scaled  [N, 24, 3]
 - prob_lstm_q10_scaled        [N, 24]
 - prob_lstm_q50_scaled        [N, 24]
 - prob_lstm_q90_scaled        [N, 24]


## Holt-Winters Forecasting Baseline

Fits a classical exponential smoothing baseline and produces scaled multi-step forecasts for the test set.


In [ ]:

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from tqdm import tqdm

def holt_winters_forecast(train_scaled, test_scaled, input_window=168, forecast_horizon=24):
    """
    Generates 24-hour ahead forecasts using Holt-Winters.
    For each test window, fits the model on the input_window and predicts forecast_horizon steps.
    Uses the scaled total_load column for consistency with other models.
    """
    test_values = test_scaled['total_load'].values
    train_values = train_scaled['total_load'].values

    # Build the same sliding windows as other models
    full_series = np.concatenate([train_values, test_values])
    train_len = len(train_values)

    # Number of test samples (same as other models)
    n_samples = len(test_values) - input_window - forecast_horizon + 1

    hw_preds = np.zeros((n_samples, forecast_horizon))

    # Fit one global model on training data and use it for all forecasts
    # (fitting per-window is too slow for 5000+ samples)
    print("Fitting Holt-Winters on training data...")
    hw_model = ExponentialSmoothing(
        train_values[-input_window * 10:],  # use last ~70 days of training
        trend='add',
        seasonal='add',
        seasonal_periods=24
    ).fit(optimized=True)

    # For each test window, refit on the input_window context
    # Sample every 24th window to keep runtime manageable (~200 fits instead of 5000)
    step_size = 24
    sampled_indices = list(range(0, n_samples, step_size))

    print(f"Generating forecasts for {len(sampled_indices)} sampled windows (step={step_size})...")
    for idx in tqdm(sampled_indices):
        context = test_values[idx : idx + input_window]
        try:
            model = ExponentialSmoothing(
                context,
                trend='add',
                seasonal='add',
                seasonal_periods=24
            ).fit(optimized=True, use_brute=False)
            hw_preds[idx] = model.forecast(forecast_horizon)
        except Exception:
            # Fallback: repeat last 24h (persistence)
            hw_preds[idx] = context[-forecast_horizon:]

    # Fill non-sampled indices with persistence forecast as fallback
    for idx in range(n_samples):
        if idx not in sampled_indices:
            context = test_values[idx : idx + input_window]
            hw_preds[idx] = context[-forecast_horizon:]

    return hw_preds


hw_preds_scaled = holt_winters_forecast(train_scaled, test_scaled)

print(f"\nHolt-Winters predictions shape: {hw_preds_scaled.shape}")
print("Stored variable: hw_preds_scaled")

Fitting Holt-Winters on training data...
Generating forecasts for 212 sampled windows (step=24)...


100%|██████████| 212/212 [00:19<00:00, 10.81it/s]


Holt-Winters predictions shape: (5070, 24)
Stored variable: hw_preds_scaled


In [ ]:
# =========================================================
# SAVE ALL MODEL WEIGHTS (.pt files)
# =========================================================

import os
os.makedirs('models', exist_ok=True)

torch.save(model_uni.state_dict(), 'models/univariate_lstm.pt')
torch.save(model_multi.state_dict(), 'models/multivariate_lstm.pt')
torch.save(model_prob_lstm.state_dict(), 'models/prob_lstm.pt')

# Save scaler with pickle for demo use
import pickle
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Baseline model weights saved to models/ folder")

# run this if you are using pt files to generate the predictions


In [1]:
import shutil, os
if os.path.exists('deep_learning_project'):
    shutil.rmtree('deep_learning_project')
    print("Deleted deep_learning_project")
else:
    print("Folder not found, nothing to delete")


Folder not found, nothing to delete


In [2]:
!git clone https://github.com/samarthsingh1/deep_learning_project

Cloning into 'deep_learning_project'...
remote: Enumerating objects: 354, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 354 (delta 37), reused 37 (delta 15), pack-reused 254 (from 2)
Receiving objects: 100% (354/354), 30.28 MiB | 39.60 MiB/s, done.
Resolving deltas: 100% (107/107), done.


In [22]:
# =========================================================
# LOAD PRE-TRAINED BASELINES AND GENERATE PREDICTIONS
# =========================================================

MODEL_DIR = '/content/deep_learning_project/models/baseline'
PRED_DIR = '/content/deep_learning_project/predictions/baseline'

# =========================================================
# UNIVARIATE LSTM — load .pt and run inference
# =========================================================
model_uni = LSTMForecaster(input_size=1, hidden_size=64, num_layers=2,
                           dropout=0.1, forecast_horizon=24).to(device)
model_uni.load_state_dict(torch.load(f'{MODEL_DIR}/univariate_lstm.pt', map_location=device))

X_test_uni = X_test[:, :, 0:1]
test_loader_uni = DataLoader(TensorDataset(X_test_uni, y_test), batch_size=BATCH_SIZE, shuffle=False)

model_uni.eval()
uni_preds = []
with torch.no_grad():
    for x_batch, y_batch in test_loader_uni:
        x_batch = x_batch.to(device)
        preds = model_uni(x_batch)
        uni_preds.append(preds.cpu().numpy())

univariate_lstm_preds_scaled = np.vstack(uni_preds)
print(f"Univariate LSTM: {univariate_lstm_preds_scaled.shape}")

# =========================================================
# MULTIVARIATE LSTM — load .pt and run inference
# =========================================================
model_multi = MultiFeatureLSTMForecaster(input_size=len(FEATURE_COLS), hidden_size=64,
                                          num_layers=2, dropout=0.1, forecast_horizon=24).to(device)
model_multi.load_state_dict(torch.load(f'{MODEL_DIR}/multivariate_lstm.pt', map_location=device))

test_loader_multi = DataLoader(TensorDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

model_multi.eval()
multi_preds = []
with torch.no_grad():
    for x_batch, y_batch in test_loader_multi:
        x_batch = x_batch.to(device)
        preds = model_multi(x_batch)
        multi_preds.append(preds.cpu().numpy())

multivariate_lstm_preds_scaled = np.vstack(multi_preds)
print(f"Multivariate LSTM: {multivariate_lstm_preds_scaled.shape}")

# =========================================================
# PROBABILISTIC LSTM — load .pt and run inference
# =========================================================
model_prob_lstm = ProbabilisticLSTM(input_size=len(FEATURE_COLS), hidden_size=64,
                                    num_layers=2, dropout=0.1, forecast_horizon=24,
                                    num_quantiles=3).to(device)
model_prob_lstm.load_state_dict(torch.load(f'{MODEL_DIR}/prob_lstm.pt', map_location=device))

model_prob_lstm.eval()
prob_preds = []
with torch.no_grad():
    for x_batch, y_batch in test_loader_multi:
        x_batch = x_batch.to(device)
        preds = model_prob_lstm(x_batch)
        prob_preds.append(preds.cpu().numpy())

prob_lstm_all_preds_scaled = np.vstack(prob_preds)
prob_lstm_q10_scaled = prob_lstm_all_preds_scaled[:, :, 0]
prob_lstm_q50_scaled = prob_lstm_all_preds_scaled[:, :, 1]
prob_lstm_q90_scaled = prob_lstm_all_preds_scaled[:, :, 2]
print(f"Probabilistic LSTM: {prob_lstm_all_preds_scaled.shape}")

# =========================================================
# HOLT-WINTERS — refit (no .pt file, statistical model)
# =========================================================
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from tqdm import tqdm

test_values = test_scaled['total_load'].values
n_samples = len(test_values) - INPUT_WINDOW - FORECAST_HORIZON + 1
hw_preds_scaled = np.zeros((n_samples, FORECAST_HORIZON))
step_size = 24
sampled_indices = list(range(0, n_samples, step_size))

print(f"Fitting Holt-Winters on {len(sampled_indices)} windows...")
for idx in tqdm(sampled_indices):
    context = test_values[idx : idx + INPUT_WINDOW]
    try:
        hw_model = ExponentialSmoothing(context, trend='add', seasonal='add',
                                        seasonal_periods=24).fit(optimized=True, use_brute=False)
        hw_preds_scaled[idx] = hw_model.forecast(FORECAST_HORIZON)
    except:
        hw_preds_scaled[idx] = context[-FORECAST_HORIZON:]

for idx in range(n_samples):
    if idx not in sampled_indices:
        hw_preds_scaled[idx] = test_values[idx + INPUT_WINDOW - FORECAST_HORIZON : idx + INPUT_WINDOW]

print(f"Holt-Winters: {hw_preds_scaled.shape}")

# =========================================================
# SAVE ALL PREDICTIONS
# =========================================================
import os
os.makedirs(PRED_DIR, exist_ok=True)

np.save(f'{PRED_DIR}/y_test_scaled.npy', y_test_scaled)
np.save(f'{PRED_DIR}/naive_persistence_preds_scaled.npy', naive_persistence_preds_scaled)
np.save(f'{PRED_DIR}/seasonal_naive_day_preds_scaled.npy', seasonal_naive_day_preds_scaled)
np.save(f'{PRED_DIR}/seasonal_naive_week_preds_scaled.npy', seasonal_naive_week_preds_scaled)
np.save(f'{PRED_DIR}/univariate_lstm_preds_scaled.npy', univariate_lstm_preds_scaled)
np.save(f'{PRED_DIR}/multivariate_lstm_preds_scaled.npy', multivariate_lstm_preds_scaled)
np.save(f'{PRED_DIR}/prob_lstm_all_preds_scaled.npy', prob_lstm_all_preds_scaled)
np.save(f'{PRED_DIR}/hw_preds_scaled.npy', hw_preds_scaled)

np.save(f'{PRED_DIR}/scaler_mean.npy', scaler.mean_)
np.save(f'{PRED_DIR}/scaler_scale.npy', scaler.scale_)

test_dates = test_scaled.index[INPUT_WINDOW : INPUT_WINDOW + len(y_test_scaled)]
pd.Series(test_dates).to_csv(f'{PRED_DIR}/test_dates.csv', index=False)

print("\nAll baseline predictions saved!")

Univariate LSTM: (5070, 24)
Multivariate LSTM: (5070, 24)
Probabilistic LSTM: (5070, 24, 3)
Fitting Holt-Winters on 212 windows...


100%|██████████| 212/212 [00:19<00:00, 10.77it/s]


Holt-Winters: (5070, 24)

All baseline predictions saved!
